In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
HF_TOKEN = os.environ.get("HF_TOKEN")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser, StrOutputParser

from pydantic import BaseModel,Field

/Users/shudhanshu/Desktop/Study Projects/GenAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:

from langchain_google_genai import ChatGoogleGenerativeAI
llm_model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")#"gemini-2.5-flash")

In [14]:

reviews = [
    """I’ve gone through five different 'ergonomic' mice in the last two years, and this is the first one that actually delivers. The 57-degree vertical angle feels super natural—it’s like shaking hands with your computer. The battery life is insane; I’ve been using it for three weeks on a single charge. If you’re a coder or designer spending all day at a desk, just buy it. Your carpal tunnel will thank you.""",
    """It’s a decent mouse for office tasks and browsing. The click is very quiet, which is great for open offices. However, I’m giving it 3 stars because the 'AeroGrip' texture is a bit of a magnet for dust and oils—I find myself wiping it down every day. Also, the polling rate is a bit slow for gaming; I noticed some lag in Call of Duty. Great for Excel, not for FPS.""",
    """I bought this for my daughter who is a digital artist, but it’s way too bulky. Even though it says 'universal fit,' her hand can barely reach the scroll wheel comfortably. The software was also a nightmare to install on her Mac. It’s a nice-looking piece of tech, but it’s definitely designed for people with large hands. We’ll be returning this and looking for a 'Mini' version.""",
    """Worked fine for exactly six days and then the sensor just died. The lights come on, but the cursor doesn't move. I tried changing the battery, updating drivers, and using a different USB port—nothing. It feels like cheap plastic junk for the price they’re charging. Don't believe the hype, stick with the big name brands."""
]

In [9]:
class QueryResponse(BaseModel):
    summary: str=Field(description="A brief summary of customer review by user")
    positive: list=Field(description="list 3 bullet points to show positive mentioned in the review")
    negative: list=Field(description="list 3 bullet points to show negatives mentioned in the review")
    sentiment: list=Field(description="one line showing sentiment of the review")
    emotion: list=Field(description="one line showing emotion of the review")
    email: str=Field(description="detailed email to the customer based on sentiment")



json_parser = JsonOutputParser(pydantic_object=QueryResponse)
json_parser

JsonOutputParser(pydantic_object=<class '__main__.QueryResponse'>)

In [10]:
prompt_txt = """

    Analyse the given customer review and generate response based on instructions
    mentioned below in the format instruction.

    Also remember to write detailed email response for the email field based on these conditions:
    -email should be addressed to Dear Customer and sighned with service agent.
    -thank them if  sentiment is positive or neutral
    -apologize if the review is negative

    format instructions:
    {format_instructions}

    review:
    {review}
"""


prompt = prompt = PromptTemplate(
    template = prompt_txt,
    input_variables=["review"],
    partial_variables={"format_instructions":json_parser.get_format_instructions()}
)

prompt

PromptTemplate(input_variables=['review'], input_types={}, partial_variables={'format_instructions': 'STRICT OUTPUT FORMAT:\n- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.\n- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).\n- Do not prepend or append any text (e.g., do not write "Here is the JSON:").\n- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output

In [12]:
chain = (
    prompt
    |
    llm_model
    |
    json_parser
)

In [15]:
reviews_formated = [{"review":review} for review in reviews]
reviews_formated

[{'review': "I’ve gone through five different 'ergonomic' mice in the last two years, and this is the first one that actually delivers. The 57-degree vertical angle feels super natural—it’s like shaking hands with your computer. The battery life is insane; I’ve been using it for three weeks on a single charge. If you’re a coder or designer spending all day at a desk, just buy it. Your carpal tunnel will thank you."},
 {'review': "It’s a decent mouse for office tasks and browsing. The click is very quiet, which is great for open offices. However, I’m giving it 3 stars because the 'AeroGrip' texture is a bit of a magnet for dust and oils—I find myself wiping it down every day. Also, the polling rate is a bit slow for gaming; I noticed some lag in Call of Duty. Great for Excel, not for FPS."},
 {'review': "I bought this for my daughter who is a digital artist, but it’s way too bulky. Even though it says 'universal fit,' her hand can barely reach the scroll wheel comfortably. The software 

In [16]:
response = chain.map().invoke(reviews_formated)
response

[{'summary': 'The customer is highly satisfied with the ergonomic mouse, noting that it successfully delivers on its promises after they tried five other failed models. They particularly praised the natural 57-degree angle and the exceptional three-week battery life, recommending it for professional coders and designers.',
  'positive': ["The 57-degree vertical angle provides a natural 'handshake' feel that is superior to other ergonomic mice.",
   'Outstanding battery life, lasting over three weeks on a single charge.',
   'Effectively alleviates carpal tunnel strain for long-term desk users like designers and coders.'],
  'negative': ["The customer mentioned a frustrating history of going through five other 'ergonomic' mice that failed to deliver.",
   'No specific negative aspects of the current product were mentioned in the review.',
   'The review does not provide information on price, which might be a factor for some buyers.'],
  'sentiment': ['Positive'],
  'emotion': ['Enthusia

In [17]:
# IT Support Analyst

In [21]:
class ItSupportResponse(BaseModel):
    orig_msg: str = Field(description="The original customer it support query message")
    orig_lang: str = Field(description="detect language of the customer message")
    category: str = Field(description="1-2 word describint the category of  the problem")
    trans_msg: str = Field(description="translated customer it support query message in english")
    response: str = Field(description="response to customer in their original language")
    trans_response: str = Field(description="response to customer in english")

json_parser = JsonOutputParser(pydantic_object=ItSupportResponse)
json_parser

JsonOutputParser(pydantic_object=<class '__main__.ItSupportResponse'>)

In [22]:
prompt_txt = """
    Act as a Information Technology(IT) support agent.
    for the it support message mentioned below
    use the following out format instructions for generating output response

    output format instructions:
    {format_instructions}

    customer it suport message:
    {message}
"""

prompt = PromptTemplate(
    template = prompt_txt,
    input_variables=["message"],
    partial_variables={"format_instructions":json_parser.get_format_instructions()}
)

prompt

PromptTemplate(input_variables=['message'], input_types={}, partial_variables={'format_instructions': 'STRICT OUTPUT FORMAT:\n- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.\n- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).\n- Do not prepend or append any text (e.g., do not write "Here is the JSON:").\n- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the outpu

In [23]:
it_chain = (
    prompt
    |
    llm_model
    |
    json_parser
)


In [24]:
it_query = "Mon ordinateur est très lent et l'utilisation du processeur est à 100 %. Comment puis-je identifier quel processus est à l'origine du problème ?"

response = it_chain.invoke({"message":it_query})
response

{'orig_msg': "Mon ordinateur est très lent et l'utilisation du processeur est à 100 %. Comment puis-je identifier quel processus est à l'origine du problème ?",
 'orig_lang': 'French',
 'category': 'Performance Issues',
 'trans_msg': 'My computer is very slow and the CPU usage is at 100%. How can I identify which process is causing the problem?',
 'response': "Bonjour. Pour identifier le processus responsable, faites un clic droit sur la barre des tâches et sélectionnez 'Gestionnaire des tâches' (ou appuyez sur Ctrl + Maj + Échap). Allez dans l'onglet 'Processus' et cliquez sur l'en-tête de la colonne 'Processeur' pour trier par utilisation décroissante. Le programme en haut de la liste est celui qui consomme le plus de ressources.",
 'trans_response': "Hello. To identify the responsible process, right-click on the taskbar and select 'Task Manager' (or press Ctrl + Shift + Esc). Go to the 'Processes' tab and click on the 'CPU' column header to sort by decreasing usage. The program at t